# SoD-Bench: real-LLM grid on a free Colab / Kaggle T4
Runs the proposal's open-weight models through vLLM's OpenAI-compatible server and the same harness used for the CPU pilot (`experiments/run_llm_pilot.py`). Every job resumes from its JSONL output, so a session cut off by usage limits loses at most one item.

In [ ]:
!git clone -b claude/sod-bench https://github.com/hbasired/agenticAI.git sodbench && cd sodbench && pip -q install -e . && pip -q install vllm openai

In [ ]:
# choose one model per session (sizes as published by the developers)
MODEL = 'Qwen/Qwen3.5-9B'   # or 'Qwen/Qwen3.5-4B', 'google/gemma-4-E4B-it', 'openai/gpt-oss-20b'
import subprocess, time
srv = subprocess.Popen(['python','-m','vllm.entrypoints.openai.api_server','--model',MODEL,'--max-model-len','8192',
                        '--gpu-memory-utilization','0.92','--quantization','bitsandbytes','--port','8000'])
time.sleep(240)

In [ ]:
%cd sodbench
!python experiments/run_llm_pilot.py --openai-base http://localhost:8000/v1 --model $MODEL \
   --out results/llm_grid_$(echo $MODEL | tr '/' '_').jsonl --configs D0,D1,D6,D6-recfirst \
   --per-family 25 --twins 60 --benign 60 --seeds 101,102,103,104

In [ ]:
!python experiments/analyze.py pilot

Prompt Guard 2 (licence-gated on Hugging Face) can replace the ProtectAI detector after accepting Meta's licence: set `HF_TOKEN` and point `OnnxInjectionDetector` at an exported ONNX copy, or score with `transformers` directly.